# Count parameters of each model to see where encoding will take longer

In [1]:
import torch
from collections import Counter

In [2]:
def load_state(path):
    # weights_only=False is needed for Lightning .ckpt files
    obj = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(obj, dict) and "state_dict" in obj:
        obj = obj["state_dict"]
    return obj

In [14]:
def count_all_params(path, depth=2):
    state = load_state(path)
    by_prefix = Counter()
    for k, v in state.items():
        if torch.is_tensor(v):
            by_prefix[".".join(k.split(".")[:depth])] += v.numel()
    for k, n in by_prefix.most_common():
        print(f"{n:>14,}  {k}")
    print(f"{sum(by_prefix.values()):>14,}  TOTAL")

In [8]:
def count_non_visual_params(path, depth=2):
    state = load_state(path)

    drop = ("image_encoder.", "img_time_logit_scale", "model.visual.", "model.logit_scale")
    state = {k: v for k, v in state.items() if torch.is_tensor(v) and not k.startswith(drop)}

    by_prefix = Counter()
    for k, v in state.items():
        by_prefix[".".join(k.split(".")[:depth])] += v.numel()
    for k, n in by_prefix.most_common():
        print(f"{n:>14,}  {k}")
    print(f"{sum(by_prefix.values()):>14,}  TOTAL")

## GTLoc

In [ ]:
count_all_params("/projects/bgtj/t_satclip/ckpts/gtloc/gtloc.pt", depth=2)

     3,149,824  location_encoder.LocEnc0
     3,149,824  location_encoder.LocEnc1
     3,149,824  location_encoder.LocEnc2
     3,149,824  time_encoder.TimeEnc0
     3,149,824  time_encoder.TimeEnc1
     3,149,824  time_encoder.TimeEnc2
       984,320  image_encoder.mlp
             1  img_time_logit_scale
    19,883,265  TOTAL


In [9]:
count_non_visual_params("/projects/bgtj/t_satclip/ckpts/gtloc/gtloc.pt", depth=2)

     3,149,824  location_encoder.LocEnc0
     3,149,824  location_encoder.LocEnc1
     3,149,824  location_encoder.LocEnc2
     3,149,824  time_encoder.TimeEnc0
     3,149,824  time_encoder.TimeEnc1
     3,149,824  time_encoder.TimeEnc2
    18,898,944  TOTAL


## SatCLIP

In [21]:
import glob

satclip_path = glob.glob(
    "/u/lbetti/.cache/huggingface/hub/models--MVRL--satclip-loc-enc-vit16-l40/snapshots/*/model.safetensors"
)[0]

In [22]:
satclip_path

'/u/lbetti/.cache/huggingface/hub/models--MVRL--satclip-loc-enc-vit16-l40/snapshots/6d6aac1f6b4b7935e5cf8084a29ba7451d69e345/model.safetensors'

In [23]:
count_all_params(satclip_path, depth=2)

     1,213,696  location.nnet
     1,213,696  TOTAL


## Temporal SatCLIP

In [ ]:
count_all_params("/projects/bgtj/t_satclip/test/ckpts/tsatclip-doy-1M.ckpt", depth=2)

    22,747,264  model.visual
       399,104  model.nnet
       399,104  model.location
             1  model.logit_scale
    23,545,473  TOTAL


In [10]:
count_non_visual_params("/projects/bgtj/t_satclip/test/ckpts/tsatclip-doy-1M.ckpt", depth=2)

       399,104  model.nnet
       399,104  model.location
       798,208  TOTAL
